# Content-Based Recommender

# Load libraries

In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz

import joblib

# load our training data

In [3]:
train = pd.read_parquet(
    "../data/processed/train.parquet"
)

print(train.shape)

(1928949, 5)


# Load the property files

In [4]:
properties_1 = pd.read_csv(
    "../data/raw/item_properties_part1.csv"
)

properties_2 = pd.read_csv(
    "../data/raw/item_properties_part2.csv"
)

print(properties_1.shape)
print(properties_2.shape)

(10999999, 4)
(9275903, 4)


# Identify training products

In [5]:
train_items = set(train["item_id"].unique())

print("Training items:", len(train_items))

Training items: 200974


# Filter metadata

In [6]:
props1_train = properties_1[
    properties_1["itemid"].isin(train_items)
].copy()

props2_train = properties_2[
    properties_2["itemid"].isin(train_items)
].copy()

print("Part 1:", props1_train.shape)
print("Part 2:", props2_train.shape)

Part 1: (4877980, 4)
Part 2: (4106838, 4)


# Combine metadata

In [7]:
expanded = pd.concat(
    [
        props1_train[["itemid", "property", "value"]],
        props2_train[["itemid", "property", "value"]]
    ],
    ignore_index=True
)

print("Raw filtered rows:", len(expanded))

Raw filtered rows: 8984818


# Split multi-value attributes

In [8]:
expanded["value"] = expanded["value"].astype(str)

expanded["value_tokens"] = expanded["value"].str.split()

expanded = expanded.explode(
    "value_tokens",
    ignore_index=True
)

# Create feature tokens

In [9]:
expanded["feature"] = (
    expanded["property"].astype(str)
    + "="
    + expanded["value_tokens"].astype(str)
)

In [10]:
expanded = expanded[
    ["itemid", "feature"]
].drop_duplicates()

# Remove extremely rare features

In [11]:
feature_frequency = (
    expanded
    .groupby("feature")["itemid"]
    .nunique()
)

MIN_FEATURE_FREQUENCY = 5

valid_features = feature_frequency[
    feature_frequency >= MIN_FEATURE_FREQUENCY
].index

expanded_filtered = expanded[
    expanded["feature"].isin(valid_features)
].copy()

print(
    "Products:",
    expanded_filtered["itemid"].nunique()
)

print(
    "Features:",
    expanded_filtered["feature"].nunique()
)

print(
    "Item-feature pairs:",
    len(expanded_filtered)
)

Products: 160670
Features: 86490
Item-feature pairs: 9148457


# Build product documents

In [12]:
product_documents = (
    expanded_filtered
    .groupby("itemid")["feature"]
    .apply(lambda x: " ".join(x))
)

print(product_documents.head())
print("Products:", len(product_documents))

itemid
4     available=0 115=n24.000 28=150169 28=176547 83...
6     categoryid=1091 28=150169 28=610517 713=373898...
15    839=245772 790=n0.000 283=433564 283=245772 28...
16    328=1042438 839=71304 28=150169 28=431733 1036...
17    790=n27120.000 227=245617 159=519769 6=245617 ...
Name: feature, dtype: object
Products: 160670


# create the sparse representation

In [13]:
vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\S+",
    min_df=5,
    dtype=np.float32
)

product_matrix = vectorizer.fit_transform(
    product_documents.values
)

In [14]:
print("Matrix shape:", product_matrix.shape)
print("Non-zero values:", product_matrix.nnz)

Matrix shape: (160670, 86490)
Non-zero values: 9148457


In [15]:
total_elements = (
    product_matrix.shape[0] *
    product_matrix.shape[1]
)

sparsity = 1 - (
    product_matrix.nnz / total_elements
)

print("Sparsity:", sparsity)
print("Density:", 1 - sparsity)

Sparsity: 0.9993416646731573
Density: 0.0006583353268426739


# Item-to-Item Content Recommendation

# Create product ID mapping

In [16]:
product_ids = product_documents.index.to_numpy()

print("Number of product IDs:", len(product_ids))
print("First 10:", product_ids[:10])

Number of product IDs: 160670
First 10: [ 4  6 15 16 17 19 22 24 25 26]


# Use cosine similarity for one product

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

In [18]:
sample_product = product_ids[0]

sample_index = 0

print("Product:", sample_product)

Product: 4


In [19]:
#calculate its similarity against all products

similarities = cosine_similarity(
    product_matrix[sample_index],
    product_matrix
).flatten()

# Get the most similar products

In [20]:
top_indices = np.argsort(
    similarities
)[::-1][:11]

In [21]:
similar_products = pd.DataFrame({
    "item_id": product_ids[top_indices],
    "similarity": similarities[top_indices]
})

print(similar_products)

    item_id  similarity
0         4    1.000000
1    411867    0.817019
2    257709    0.714949
3     44197    0.682005
4    303369    0.611073
5    228565    0.571240
6    214918    0.558065
7     49533    0.538350
8     74291    0.524194
9    394394    0.489144
10   388905    0.478749


# Remove the product itself

In [22]:
similar_products = similar_products[
    similar_products["item_id"] != sample_product
].head(10)

print(similar_products)

    item_id  similarity
1    411867    0.817019
2    257709    0.714949
3     44197    0.682005
4    303369    0.611073
5    228565    0.571240
6    214918    0.558065
7     49533    0.538350
8     74291    0.524194
9    394394    0.489144
10   388905    0.478749


# Create a reusable function

In [23]:
def similar_items(item_id, k=10):

    if item_id not in product_id_to_index:
        return pd.DataFrame(
            columns=["item_id", "similarity"]
        )

    idx = product_id_to_index[item_id]

    similarities = cosine_similarity(
        product_matrix[idx],
        product_matrix
    ).flatten()

    similarities[idx] = -1

    top_indices = np.argpartition(
        similarities,
        -k
    )[-k:]

    top_indices = top_indices[
        np.argsort(
            similarities[top_indices]
        )[::-1]
    ]

    return pd.DataFrame({
        "item_id": product_ids[top_indices],
        "similarity": similarities[top_indices]
    })

In [24]:
product_id_to_index = {
    item_id: idx
    for idx, item_id in enumerate(product_ids)
}

# Testing

In [25]:
sample_product = int(product_ids[0])

print(
    similar_items(
        sample_product,
        k=10
    )
)

   item_id  similarity
0   411867    0.817019
1   257709    0.714949
2    44197    0.682005
3   303369    0.611073
4   228565    0.571240
5   214918    0.558065
6    49533    0.538350
7    74291    0.524194
8   394394    0.489144
9   388905    0.478749


# Save the content model artifacts

In [26]:
import joblib
from scipy.sparse import save_npz

joblib.dump(
    vectorizer,
    "../models/content_vectorizer.joblib"
)

np.save(
    "../models/product_ids.npy",
    product_ids
)

save_npz(
    "../models/product_tfidf.npz",
    product_matrix
)

## Personalized Content-Based Recommender

# Create the user profile

In [27]:
# Load validation

validation = pd.read_parquet(
    "../data/processed/validation.parquet"
)

In [28]:
product_id_to_index = {
    item_id: idx
    for idx, item_id in enumerate(product_ids)
}

In [29]:
# keep only training interactions whose products have metadata

train_content = train[
    train["item_id"].isin(product_id_to_index)
].copy()

print("Training interactions:", len(train_content))
print(
    "Unique users:",
    train_content["user_id"].nunique()
)

Training interactions: 1754655
Unique users: 859881


# Build a weighted user profile

In [30]:
sample_user = train_content["user_id"].value_counts().index[0]

user_history = train_content[
    train_content["user_id"] == sample_user
]

print(user_history)

         user_id  item_id        event               timestamp  \
815233   1150086   133542         view 2015-06-11 14:55:17.389   
815538   1150086   167873         view 2015-06-11 15:15:35.608   
815606   1150086   231726         view 2015-06-11 15:20:06.651   
815669   1150086   427777         view 2015-06-11 15:24:48.806   
815675   1150086   398115         view 2015-06-11 15:25:32.155   
...          ...      ...          ...                     ...   
1928646  1150086    60966         view 2015-08-02 23:10:32.951   
1928661  1150086    60966    addtocart 2015-08-02 23:11:21.322   
1928677  1150086    60966  transaction 2015-08-02 23:12:39.072   
1928757  1150086    60966         view 2015-08-02 23:17:24.572   
1928858  1150086     2455         view 2015-08-02 23:24:11.439   

         interaction_strength  
815233                      1  
815538                      1  
815606                      1  
815669                      1  
815675                      1  
...            

In [31]:
print(
    "Number of interacted products:",
    user_history["item_id"].nunique()
)

Number of interacted products: 2229


# Construct that user's profile

In [32]:
user_indices = []
user_weights = []

for _, row in user_history.iterrows():

    item_id = row["item_id"]

    if item_id in product_id_to_index:
        user_indices.append(
            product_id_to_index[item_id]
        )
        user_weights.append(
            row["interaction_strength"]
        )

In [33]:
user_weights = np.array(
    user_weights,
    dtype=np.float32
)

In [34]:
user_item_matrix = product_matrix[
    user_indices
]

In [35]:
weighted_profile = (
    user_item_matrix.multiply(
        user_weights[:, None]
    ).sum(axis=0)
)

weighted_profile = np.asarray(
    weighted_profile
)

# Normalize the profile

In [36]:
from sklearn.preprocessing import normalize

user_profile = normalize(
    weighted_profile
)

# Find similar products

In [37]:
user_similarities = cosine_similarity(
    user_profile,
    product_matrix
).flatten()

# Remove products the user already saw

In [38]:
seen_items = set(
    user_history["item_id"]
)

In [39]:
for item_id in seen_items:

    if item_id in product_id_to_index:

        idx = product_id_to_index[item_id]

        user_similarities[idx] = -1

In [40]:
#retrieve Top-10

top_indices = np.argpartition(
    user_similarities,
    -10
)[-10:]

top_indices = top_indices[
    np.argsort(
        user_similarities[top_indices]
    )[::-1]
]

In [41]:
#Create the recommendation table

recommendations = pd.DataFrame({
    "item_id": product_ids[top_indices],
    "score": user_similarities[top_indices]
})

print("User:", sample_user)
print(recommendations)

User: 1150086
   item_id     score
0    47180  0.310817
1   352748  0.294132
2   333719  0.293902
3   352030  0.293469
4    61203  0.292449
5   116055  0.290111
6   250880  0.289164
7   120798  0.288648
8    68047  0.288602
9   160250  0.288256


# Get warm validation users

In [42]:
train_users = set(train["user_id"].unique())

validation_users = set(
    validation["user_id"].unique()
)

warm_validation_users = (
    validation_users & train_users
)

print(
    "Warm validation users:",
    len(warm_validation_users)
)

Warm validation users: 18965


# Build validation ground truth

In [43]:
validation_ground_truth = (
    validation
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

In [44]:
warm_ground_truth = {
    user_id: items
    for user_id, items
    in validation_ground_truth.items()
    if user_id in warm_validation_users
}

print(
    "Users in evaluation:",
    len(warm_ground_truth)
)

Users in evaluation: 18965


# Create the reusable recommender

In [45]:
def recommend_content(
    user_id,
    k=10
):
    user_history = train_content[
        train_content["user_id"] == user_id
    ]

    user_indices = []
    user_weights = []

    for _, row in user_history.iterrows():

        item_id = row["item_id"]

        if item_id in product_id_to_index:

            user_indices.append(
                product_id_to_index[item_id]
            )

            user_weights.append(
                row["interaction_strength"]
            )

    if len(user_indices) == 0:
        return []

    weights = np.asarray(
        user_weights,
        dtype=np.float32
    )

    user_item_matrix = product_matrix[
        user_indices
    ]

    profile = user_item_matrix.multiply(
        weights[:, None]
    ).sum(axis=0)

    profile = normalize(
        np.asarray(profile)
    )

    scores = cosine_similarity(
        profile,
        product_matrix
    ).ravel()

    # Remove previously interacted products
    seen_items = set(
        user_history["item_id"]
    )

    for item_id in seen_items:

        idx = product_id_to_index.get(item_id)

        if idx is not None:
            scores[idx] = -1

    top_indices = np.argpartition(
        scores,
        -k
    )[-k:]

    top_indices = top_indices[
        np.argsort(
            scores[top_indices]
        )[::-1]
    ]

    return product_ids[top_indices].tolist()

# Batch Content-Based Evaluation

# Build the warm-user training histories

In [46]:
warm_train = train_content[
    train_content["user_id"].isin(
        warm_validation_users
    )
].copy()

print("Warm-user training interactions:", len(warm_train))
print(
    "Warm users:",
    warm_train["user_id"].nunique()
)

Warm-user training interactions: 109022
Warm users: 17555


# Create a user → index mapping

In [47]:
warm_user_ids = np.array(
    sorted(warm_validation_users)
)

user_id_to_index = {
    user_id: idx
    for idx, user_id in enumerate(warm_user_ids)
}

print("Users:", len(warm_user_ids))

Users: 18965


# Build sparse user profiles

In [48]:
from scipy.sparse import csr_matrix

In [49]:
rows = []
cols = []
weights = []

for row in warm_train.itertuples(index=False):

    user_idx = user_id_to_index[row.user_id]

    item_idx = product_id_to_index.get(
        row.item_id
    )

    if item_idx is not None:
        rows.append(user_idx)
        cols.append(item_idx)
        weights.append(
            row.interaction_strength
        )

In [50]:
interaction_matrix = csr_matrix(
    (
        np.asarray(weights, dtype=np.float32),
        (
            np.asarray(rows),
            np.asarray(cols)
        )
    ),
    shape=(
        len(warm_user_ids),
        product_matrix.shape[0]
    ),
    dtype=np.float32
)

# Aggregate repeated interactions

In [51]:
print("User-item matrix shape:")
print(interaction_matrix.shape)

print(
    "Non-zero interactions:",
    interaction_matrix.nnz
)

User-item matrix shape:
(18965, 160670)
Non-zero interactions: 68276


# Convert interaction weights into user profiles

In [52]:
user_profiles = (
    interaction_matrix @ product_matrix
)

In [53]:
user_profiles = normalize(
    user_profiles,
    norm="l2",
    axis=1
)

In [54]:
print(
    "User profile matrix:",
    user_profiles.shape
)

print(
    "Non-zero values:",
    user_profiles.nnz
)

User profile matrix: (18965, 86490)
Non-zero values: 2434788


# evaluate in batches

In [55]:
BATCH_SIZE = 256

# First test one batch

In [56]:
batch_profiles = user_profiles[
    :256
]

batch_scores = (
    batch_profiles @ product_matrix.T
)

print(
    "Batch score matrix:",
    batch_scores.shape
)

Batch score matrix: (256, 160670)


# Build evaluation ground truth

In [57]:
ground_truth_lists = [
    warm_ground_truth.get(
        user_id,
        set()
    )
    for user_id in warm_user_ids
]

# Build training seen-items

In [58]:
seen_items_by_user = (
    warm_train
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

# Evaluation functions

In [59]:
def precision_at_k(recommended, relevant, k):
    if k == 0:
        return 0.0

    hits = sum(
        item in relevant
        for item in recommended[:k]
    )

    return hits / k

In [60]:
def recall_at_k(recommended, relevant, k):
    if len(relevant) == 0:
        return 0.0

    hits = sum(
        item in relevant
        for item in recommended[:k]
    )

    return hits / len(relevant)

In [61]:
def hit_rate_at_k(recommended, relevant, k):
    return float(
        any(
            item in relevant
            for item in recommended[:k]
        )
    )

In [62]:
def ndcg_at_k(recommended, relevant, k):

    if len(relevant) == 0:
        return 0.0

    dcg = 0.0

    for rank, item in enumerate(
        recommended[:k],
        start=1
    ):
        if item in relevant:
            dcg += 1 / np.log2(rank + 1)

    ideal_hits = min(
        len(relevant),
        k
    )

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(
            1,
            ideal_hits + 1
        )
    )

    return dcg / idcg

# Create the batch evaluator

In [63]:
def evaluate_content_model(
    user_profiles,
    batch_size=256,
    ks=(5, 10, 20)
):

    metrics = {
        k: {
            "precision": [],
            "recall": [],
            "ndcg": [],
            "hit": []
        }
        for k in ks
    }

    n_users = user_profiles.shape[0]

    for start in range(
        0,
        n_users,
        batch_size
    ):

        end = min(
            start + batch_size,
            n_users
        )

        print(
            f"Processing users "
            f"{start}:{end}"
        )

        batch_profiles = user_profiles[
            start:end
        ]

        scores = (
            batch_profiles @ product_matrix.T
        )

        scores = scores.toarray()

        # Remove training-seen items
        for local_idx, global_idx in enumerate(
            range(start, end)
        ):

            user_id = warm_user_ids[
                global_idx
            ]

            seen_items = (
                seen_items_by_user
                .get(user_id, set())
            )

            for item_id in seen_items:

                product_idx = (
                    product_id_to_index
                    .get(item_id)
                )

                if product_idx is not None:
                    scores[
                        local_idx,
                        product_idx
                    ] = -np.inf

        # Top max K
        max_k = max(ks)

        top_indices = np.argpartition(
            scores,
            -max_k,
            axis=1
        )[:, -max_k:]

        # Sort those candidates
        row_indices = np.arange(
            end - start
        )[:, None]

        top_scores = scores[
            row_indices,
            top_indices
        ]

        order = np.argsort(
            top_scores,
            axis=1
        )[:, ::-1]

        top_indices = np.take_along_axis(
            top_indices,
            order,
            axis=1
        )

        # Evaluate
        for local_idx, global_idx in enumerate(
            range(start, end)
        ):

            user_id = warm_user_ids[
                global_idx
            ]

            relevant = ground_truth_lists[
                global_idx
            ]

            recommendations = [
                product_ids[idx]
                for idx in top_indices[
                    local_idx
                ]
            ]

            for k in ks:

                metrics[k]["precision"].append(
                    precision_at_k(
                        recommendations,
                        relevant,
                        k
                    )
                )

                metrics[k]["recall"].append(
                    recall_at_k(
                        recommendations,
                        relevant,
                        k
                    )
                )

                metrics[k]["ndcg"].append(
                    ndcg_at_k(
                        recommendations,
                        relevant,
                        k
                    )
                )

                metrics[k]["hit"].append(
                    hit_rate_at_k(
                        recommendations,
                        relevant,
                        k
                    )
                )

    results = []

    for k in ks:

        results.append({
            "Model": "Content-Based",
            "K": k,
            "Precision@K": np.mean(
                metrics[k]["precision"]
            ),
            "Recall@K": np.mean(
                metrics[k]["recall"]
            ),
            "NDCG@K": np.mean(
                metrics[k]["ndcg"]
            ),
            "HitRate@K": np.mean(
                metrics[k]["hit"]
            )
        })

    return pd.DataFrame(results)

# Run the evaluation

In [64]:
content_results = evaluate_content_model(
    user_profiles,
    batch_size=256,
    ks=(5, 10, 20)
)

print(content_results)

Processing users 0:256


Processing users 256:512
Processing users 512:768
Processing users 768:1024
Processing users 1024:1280
Processing users 1280:1536
Processing users 1536:1792
Processing users 1792:2048
Processing users 2048:2304
Processing users 2304:2560
Processing users 2560:2816
Processing users 2816:3072
Processing users 3072:3328
Processing users 3328:3584
Processing users 3584:3840
Processing users 3840:4096
Processing users 4096:4352
Processing users 4352:4608
Processing users 4608:4864
Processing users 4864:5120
Processing users 5120:5376
Processing users 5376:5632
Processing users 5632:5888
Processing users 5888:6144
Processing users 6144:6400
Processing users 6400:6656
Processing users 6656:6912
Processing users 6912:7168
Processing users 7168:7424
Processing users 7424:7680
Processing users 7680:7936
Processing users 7936:8192
Processing users 8192:8448
Processing users 8448:8704
Processing users 8704:8960
Processing users 8960:9216
Processing users 9216:9472
Processing users 9472:9728
Proces

# compare with Model 0

In [65]:
baseline_results = pd.DataFrame([
    {
        "Model": "Popularity",
        "K": 5,
        "Precision@K": 0.000736,
        "Recall@K": 0.002944,
        "NDCG@K": 0.001915,
        "HitRate@K": 0.003677
    },
    {
        "Model": "Popularity",
        "K": 10,
        "Precision@K": 0.000617,
        "Recall@K": 0.004861,
        "NDCG@K": 0.002541,
        "HitRate@K": 0.006058
    },
    {
        "Model": "Popularity",
        "K": 20,
        "Precision@K": 0.000540,
        "Recall@K": 0.008072,
        "NDCG@K": 0.003396,
        "HitRate@K": 0.010523
    }
])

comparison = pd.concat(
    [baseline_results, content_results],
    ignore_index=True
)

print(comparison)

           Model   K  Precision@K  Recall@K    NDCG@K  HitRate@K
0     Popularity   5     0.000736  0.002944  0.001915   0.003677
1     Popularity  10     0.000617  0.004861  0.002541   0.006058
2     Popularity  20     0.000540  0.008072  0.003396   0.010523
3  Content-Based   5     0.005610  0.016306  0.012629   0.025415
4  Content-Based  10     0.004097  0.023437  0.015026   0.036172
5  Content-Based  20     0.002839  0.031465  0.017325   0.048194


In [73]:

# RECREATE BPR-ELIGIBLE COHORT

# 1. Aggregate training interactions
user_item = (
    train
    .groupby(["user_id", "item_id"])
    ["interaction_strength"]
    .sum()
    .reset_index()
)

# 2. Count interactions per user/item
user_counts = (
    user_item
    .groupby("user_id")
    .size()
)

item_counts = (
    user_item
    .groupby("item_id")
    .size()
)

# 3. Apply the SAME thresholds used for BPR
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5

active_users = set(
    user_counts[
        user_counts >= MIN_USER_INTERACTIONS
    ].index
)

active_items = set(
    item_counts[
        item_counts >= MIN_ITEM_INTERACTIONS
    ].index
)

print("Active users:", len(active_users))
print("Active items:", len(active_items))

# 4. Validation users that BPR could potentially evaluate
bpr_validation_users = (
    set(validation["user_id"].unique())
    & active_users
)

# 5. Validation interactions must also be on BPR-known items
fair_validation = validation[
    validation["user_id"].isin(
        bpr_validation_users
    )
    &
    validation["item_id"].isin(
        active_items
    )
].copy()

# 6. Ground truth
fair_ground_truth = (
    fair_validation
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

# 7. Final fair evaluation users
fair_users = np.array(
    sorted(fair_ground_truth.keys())
)

print("\nFair evaluation users:", len(fair_users))
print(
    "Fair validation interactions:",
    len(fair_validation)
)

Active users: 28009
Active items: 63115

Fair evaluation users: 2174
Fair validation interactions: 19661


In [74]:
content_user_indices = {
    user_id: idx
    for idx, user_id in enumerate(warm_user_ids)
}

fair_users = np.array([
    user_id
    for user_id in fair_users
    if user_id in content_user_indices
])

print(
    "Users available in Content model:",
    len(fair_users)
)

Users available in Content model: 2174


In [75]:
fair_user_indices = [
    content_user_indices[user_id]
    for user_id in fair_users
]

fair_profiles = user_profiles[
    fair_user_indices
]

print(
    "Fair profile matrix:",
    fair_profiles.shape
)

Fair profile matrix: (2174, 86490)


In [76]:
bpr_eval_users = np.load(
    "../data/processed/bpr_eval_users.npy"
)

print(
    "Exact BPR evaluation users:",
    len(bpr_eval_users)
)

Exact BPR evaluation users: 2047


In [77]:
content_user_indices = {
    user_id: idx
    for idx, user_id in enumerate(warm_user_ids)
}

fair_users = np.array([
    user_id
    for user_id in bpr_eval_users
    if user_id in content_user_indices
])

print(
    "Users available in both models:",
    len(fair_users)
)

Users available in both models: 2047


# Build the exact Content ground truth

In [78]:
fair_validation = validation[
    validation["user_id"].isin(fair_users)
].copy()

fair_ground_truth = (
    fair_validation
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()

)

print("Fair users:", len(fair_ground_truth))
print(
    "Fair validation interactions:",
    len(fair_validation)
)

Fair users: 2047
Fair validation interactions: 25549


# Get Content profiles

In [79]:
content_user_indices = {
    user_id: idx
    for idx, user_id in enumerate(warm_user_ids)
}

fair_user_indices = [
    content_user_indices[user_id]
    for user_id in fair_users
]

fair_profiles = user_profiles[
    fair_user_indices
]

print(
    "Fair profile matrix:",
    fair_profiles.shape
)

Fair profile matrix: (2047, 86490)


In [81]:
def get_content_recommendations_for_users(
    profiles,
    users,
    k=20,
    batch_size=256
):
    all_recommendations = {}

    for start in range(
        0,
        len(users),
        batch_size
    ):

        end = min(
            start + batch_size,
            len(users)
        )

        print(
            f"Processed {start}:{end}"
        )

        # User profile × item TF-IDF
        scores = (
            profiles[start:end]
            @ product_matrix.T
        ).toarray()

        for local_idx, user_id in enumerate(
            users[start:end]
        ):

            # Remove items already seen in training
            seen = seen_items_by_user.get(
                user_id,
                set()
            )

            for item_id in seen:

                item_idx = product_id_to_index.get(
                    item_id
                )

                if item_idx is not None:
                    scores[
                        local_idx,
                        item_idx
                    ] = -np.inf

            # Top-K
            top_indices = np.argpartition(
                scores[local_idx],
                -k
            )[-k:]

            # Sort by score
            top_indices = top_indices[
                np.argsort(
                    scores[
                        local_idx,
                        top_indices
                    ]
                )[::-1]
            ]

            all_recommendations[user_id] = [
                product_ids[idx]
                for idx in top_indices
            ]

    return all_recommendations

# Generate Content recommendations

In [82]:
fair_content_recs = (
    get_content_recommendations_for_users(
        fair_profiles,
        fair_users,
        k=20
    )
)

Processed 0:256
Processed 256:512
Processed 512:768
Processed 768:1024
Processed 1024:1280
Processed 1280:1536
Processed 1536:1792
Processed 1792:2047


# Evaluate Content

In [83]:
fair_content_metrics = {
    5: {
        "precision": [],
        "recall": [],
        "ndcg": [],
        "hit": []
    },
    10: {
        "precision": [],
        "recall": [],
        "ndcg": [],
        "hit": []
    },
    20: {
        "precision": [],
        "recall": [],
        "ndcg": [],
        "hit": []
    }
}

for user_id in fair_users:

    recommended = fair_content_recs[user_id]

    relevant = fair_ground_truth.get(
        user_id,
        set()
    )

    for k in [5, 10, 20]:

        fair_content_metrics[k]["precision"].append(
            precision_at_k(
                recommended,
                relevant,
                k
            )
        )

        fair_content_metrics[k]["recall"].append(
            recall_at_k(
                recommended,
                relevant,
                k
            )
        )

        fair_content_metrics[k]["ndcg"].append(
            ndcg_at_k(
                recommended,
                relevant,
                k
            )
        )

        fair_content_metrics[k]["hit"].append(
            hit_rate_at_k(
                recommended,
                relevant,
                k
            )
        )

In [84]:
fair_content_results = pd.DataFrame([
    {
        "Model": "Content-Based",
        "K": k,
        "Precision@K": np.mean(
            fair_content_metrics[k]["precision"]
        ),
        "Recall@K": np.mean(
            fair_content_metrics[k]["recall"]
        ),
        "NDCG@K": np.mean(
            fair_content_metrics[k]["ndcg"]
        ),
        "HitRate@K": np.mean(
            fair_content_metrics[k]["hit"]
        )
    }
    for k in [5, 10, 20]
])

print(fair_content_results)

           Model   K  Precision@K  Recall@K    NDCG@K  HitRate@K
0  Content-Based   5     0.008891  0.011441  0.011852   0.037616
1  Content-Based  10     0.007328  0.019636  0.014270   0.056668
2  Content-Based  20     0.005423  0.026594  0.016558   0.076698


In [87]:
bpr_results = pd.DataFrame([
    {
        "Model": "BPR",
        "K": 5,
        "Precision@K": 0.003420,
        "Recall@K": 0.003989,
        "NDCG@K": 0.004817,
        "HitRate@K": 0.014656
    },
    {
        "Model": "BPR",
        "K": 10,
        "Precision@K": 0.002687,
        "Recall@K": 0.006208,
        "NDCG@K": 0.005380,
        "HitRate@K": 0.021006
    },
    {
        "Model": "BPR",
        "K": 20,
        "Precision@K": 0.002003,
        "Recall@K": 0.009472,
        "NDCG@K": 0.006367,
        "HitRate@K": 0.027357
    }
])

In [88]:
fair_comparison = pd.concat(
    [
        fair_content_results,
        bpr_results
    ],
    ignore_index=True
)

print(fair_comparison)

           Model   K  Precision@K  Recall@K    NDCG@K  HitRate@K
0  Content-Based   5     0.008891  0.011441  0.011852   0.037616
1  Content-Based  10     0.007328  0.019636  0.014270   0.056668
2  Content-Based  20     0.005423  0.026594  0.016558   0.076698
3            BPR   5     0.003420  0.003989  0.004817   0.014656
4            BPR  10     0.002687  0.006208  0.005380   0.021006
5            BPR  20     0.002003  0.009472  0.006367   0.027357


In [89]:
# Global popularity from TRAIN only

popularity_ranking = (
    train
    .groupby("item_id")
    ["interaction_strength"]
    .sum()
    .sort_values(ascending=False)
    .index
    .to_numpy()
)

# Remove items already seen by each user
fair_popularity_recs = {}

for user_id in fair_users:

    seen = set(
        train[
            train["user_id"] == user_id
        ]["item_id"]
    )

    recommendations = [
        item
        for item in popularity_ranking
        if item not in seen
    ][:20]

    fair_popularity_recs[user_id] = recommendations

print(
    "Users with popularity recommendations:",
    len(fair_popularity_recs)
)

Users with popularity recommendations: 2047


In [90]:
fair_popularity_metrics = {
    5: {"precision": [], "recall": [], "ndcg": [], "hit": []},
    10: {"precision": [], "recall": [], "ndcg": [], "hit": []},
    20: {"precision": [], "recall": [], "ndcg": [], "hit": []}
}

for user_id in fair_users:

    recommended = fair_popularity_recs[user_id]

    relevant = fair_ground_truth.get(
        user_id,
        set()
    )

    for k in [5, 10, 20]:

        fair_popularity_metrics[k]["precision"].append(
            precision_at_k(
                recommended,
                relevant,
                k
            )
        )

        fair_popularity_metrics[k]["recall"].append(
            recall_at_k(
                recommended,
                relevant,
                k
            )
        )

        fair_popularity_metrics[k]["ndcg"].append(
            ndcg_at_k(
                recommended,
                relevant,
                k
            )
        )

        fair_popularity_metrics[k]["hit"].append(
            hit_rate_at_k(
                recommended,
                relevant,
                k
            )
        )

In [91]:
fair_popularity_results = pd.DataFrame([
    {
        "Model": "Popularity",
        "K": k,
        "Precision@K": np.mean(
            fair_popularity_metrics[k]["precision"]
        ),
        "Recall@K": np.mean(
            fair_popularity_metrics[k]["recall"]
        ),
        "NDCG@K": np.mean(
            fair_popularity_metrics[k]["ndcg"]
        ),
        "HitRate@K": np.mean(
            fair_popularity_metrics[k]["hit"]
        )
    }
    for k in [5, 10, 20]
])

print(fair_popularity_results)

        Model   K  Precision@K  Recall@K    NDCG@K  HitRate@K
0  Popularity   5     0.002638  0.002970  0.004131   0.011724
1  Popularity  10     0.002003  0.003527  0.003936   0.014167
2  Popularity  20     0.002076  0.006266  0.004686   0.025892


In [92]:
final_fair_comparison = pd.concat(
    [
        fair_popularity_results,
        fair_content_results,
        bpr_results
    ],
    ignore_index=True
)

print(final_fair_comparison)

           Model   K  Precision@K  Recall@K    NDCG@K  HitRate@K
0     Popularity   5     0.002638  0.002970  0.004131   0.011724
1     Popularity  10     0.002003  0.003527  0.003936   0.014167
2     Popularity  20     0.002076  0.006266  0.004686   0.025892
3  Content-Based   5     0.008891  0.011441  0.011852   0.037616
4  Content-Based  10     0.007328  0.019636  0.014270   0.056668
5  Content-Based  20     0.005423  0.026594  0.016558   0.076698
6            BPR   5     0.003420  0.003989  0.004817   0.014656
7            BPR  10     0.002687  0.006208  0.005380   0.021006
8            BPR  20     0.002003  0.009472  0.006367   0.027357


In [93]:
from scipy.sparse import save_npz
import numpy as np
import os

os.makedirs("../models", exist_ok=True)

save_npz(
    "../models/user_profiles.npz",
    user_profiles
)

np.save(
    "../models/warm_user_ids.npy",
    np.array(warm_user_ids)
)

print("Content artifacts saved.")
print("User profiles:", user_profiles.shape)
print("Warm users:", len(warm_user_ids))

Content artifacts saved.
User profiles: (18965, 86490)
Warm users: 18965
